<a href="https://colab.research.google.com/github/DarshChoudhary-09/End-to-End-Project/blob/main/Online_retail_Pyhton_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving online_retail_clean.csv to online_retail_clean.csv


In [ ]:
import pandas as pd

df = pd.read_csv('online_retail_clean.csv')
print(df.shape)
print(df.columns.tolist())
df.head()

/tmp/ipykernel_1383/3237729469.py:3: DtypeWarning: Columns (3,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('online_retail_clean.csv')


(1033036, 8)
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer_ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7\ SINGLE SIZE """,48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


In [ ]:
import pandas as pd

df = pd.read_csv('online_retail_clean.csv', escapechar='\\', quotechar='"',
                 dtype=str, na_values=['NULL', ''])
print(df.shape)
print(df.columns.tolist())

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Quantity'] = pd.to_numeric(df['Quantity'])
df['Price'] = pd.to_numeric(df['Price'])
df['Customer_ID'] = pd.to_numeric(df['Customer_ID']).astype('Int64')

df.info()

(1033036, 8)
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer_ID', 'Country']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1033036 non-null  object        
 1   StockCode    1033036 non-null  object        
 2   Description  1028761 non-null  object        
 3   Quantity     1033036 non-null  int64         
 4   InvoiceDate  1033036 non-null  datetime64[ns]
 5   Price        1033036 non-null  float64       
 6   Customer_ID  797885 non-null   Int64         
 7   Country      1033036 non-null  object        
dtypes: Int64(1), datetime64[ns](1), float64(1), int64(1), object(4)
memory usage: 64.0+ MB


In [ ]:
# FILTER AND CALCULATE RECENCY, FREQUENCY, MONETARY
rfm_data = df[(df['Customer_ID'].notna()) & (df['Quantity'] > 0)].copy()
snapshot_date = pd.to_datetime('2011-12-10')

rfm = rfm_data.groupby('Customer_ID').agg(
    RECENCY=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    FREQUENCY=('Invoice', 'nunique'),
    MONETARY=('Quantity', lambda x: (x * rfm_data.loc[x.index, 'Price']).sum())
).reset_index()

print(rfm.shape)
rfm.head()

(5881, 4)


,Customer_ID,RECENCY,FREQUENCY,MONETARY
0,12346,325,12,77556.46
1,12347,2,8,4921.53
2,12348,75,5,2019.40
3,12349,18,4,4428.69
4,12350,310,1,334.40


In [ ]:
print(rfm.columns.tolist())

['Customer_ID', 'RECENCY', 'FREQUENCY', 'MONETARY', 'R_Score', 'F_Score', 'M_Score', 'Segment']


In [ ]:
# SCORE CUSTOMER AND ASSIGN BUSINESS SEGMENTS
rfm['R_Score'] = pd.qcut(rfm['RECENCY'], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['FREQUENCY'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['MONETARY'], 5, labels=[1, 2, 3, 4, 5]).astype(int)

def segment_customer(row):
  if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
    return 'Champions'
  elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
    return 'Loyal'
  elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
    return 'At Risk'
  elif row['R_Score'] >= 4 and row['F_Score'] <= 2:
    return 'New'
  elif row['R_Score'] == 3 and row['F_Score'] <= 2:
    return 'Needs Attention'
  else:
    return 'Lost'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)
rfm['IsChurned'] = rfm['RECENCY'] > 90

print(rfm.shape)
rfm.head()


(5881, 9)


,Customer_ID,RECENCY,FREQUENCY,MONETARY,R_Score,F_Score,M_Score,Segment,IsChurned
0,12346,325,12,77556.46,2,5,5,At Risk,True
1,12347,2,8,4921.53,5,4,5,Champions,False
2,12348,75,5,2019.40,3,4,4,Loyal,False
3,12349,18,4,4428.69,5,3,5,Loyal,False
4,12350,310,1,334.40,2,1,2,Lost,True


In [ ]:
# MERGE RFM RESULTS BAACK INTO TRANSACTION AND EXPORT
df = df[(df['Customer_ID'].notna()) & (df['Quantity'] > 0)].copy()
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
df['Revenue'] = df['Quantity'] * df['Price']

df_final = df.merge(
    rfm[['Customer_ID', 'RECENCY', 'FREQUENCY', 'MONETARY', 'R_Score', 'F_Score', 'M_Score', 'Segment', 'IsChurned']],
    on='Customer_ID' , how='left'
)

df_final.to_excel('EcommerceDashboardData.xlsx', index=False)